# Basic readout: aggregating `analytics` hourly buckets

There is no `GET /stats` endpoint yet (see `scw_js/analytics-implementation-plan.md`
— reads will be owner-signature-gated, like the Growth API). This notebook is a
throwaway prototype of the aggregation logic: list one **day's** hourly objects by
prefix, sum `hits`, merge `pages`. It also doubles as a rough draft for the future
growth-agent rollup (`growth-agent/website_analytics.json`, scoped separately) —
not production code, not wired into any endpoint.

**Day-prefix, not the bare site prefix**: `list_keys`/`listObjects` is one
unpaginated page (`shared/s3-utils/src/index.ts:280-286`), so listing the whole
`counts/fretchen.eu/` prefix would silently truncate past ~41 days of hourly objects.

In [ ]:
import os
from datetime import date, datetime, timezone

from dotenv import load_dotenv

from storage import LocalStorage, S3Storage

load_dotenv()

In [ ]:
def aggregate_day(storage, site: str, day: date) -> tuple[int, dict[str, int]]:
    """Sum hits and merge pages across one UTC day's hourly buckets."""
    prefix = f"counts/{site}/{day:%Y-%m-%d}T"
    total_hits = 0
    pages: dict[str, int] = {}
    for key in storage.list_keys(prefix):
        bucket = storage.read(key)
        if not bucket:
            continue
        total_hits += bucket.get("hits", 0)
        for path, count in bucket.get("pages", {}).items():
            pages[path] = pages.get(path, 0) + count
    return total_hits, pages

## 1. Fixture pass — validate the logic against known data

No credentials needed. Writes three synthetic hourly buckets, then checks
`aggregate_day` produces the hand-computed totals.

In [ ]:
local = LocalStorage()
fixture_day = date(2026, 8, 10)

local.write("counts/fixture-site/2026-08-10T00.json", {"hits": 5, "pages": {"/a": 3, "/b": 2}})
local.write("counts/fixture-site/2026-08-10T01.json", {"hits": 2, "pages": {"/a": 1, "/c": 1}})
local.write("counts/fixture-site/2026-08-10T02.json", {"hits": 4, "pages": {"/b": 4}})

fixture_hits, fixture_pages = aggregate_day(local, "fixture-site", fixture_day)
print(fixture_hits, fixture_pages)

assert fixture_hits == 11  # 5 + 2 + 4
assert fixture_pages == {"/a": 4, "/b": 6, "/c": 1}

## 2. Real readout

Will be small/empty until `01_smoke_test.ipynb` and the frontend wiring (PR2)
have generated real traffic.

In [ ]:
s3 = S3Storage(
    access_key=os.environ["SCW_ACCESS_KEY"],
    secret_key=os.environ["SCW_SECRET_KEY"],
)
today = datetime.now(timezone.utc).date()

total_hits, pages = aggregate_day(s3, "fretchen.eu", today)
print(f"{today}: {total_hits} hits across {len(pages)} distinct pages")

In [ ]:
# top pages, most-hit first
for path, count in sorted(pages.items(), key=lambda item: -item[1])[:10]:
    print(f"{count:>6}  {path}")

---

## 3. Range readout from monthly rollups

Section 2 above reads one day out of the hourly buckets. That does not scale to
a dashboard: `listObjects` is a single un-paginated ListObjectsV2 (max 1000
keys) and a 30-day window would be 720 sequential GETs.

The rollup layer fixes both. `rollup/{site}/{YYYY-MM}.json` holds a per-day
`{hits, pages, source}` for the whole month; the keys for a date range are
**computed**, never listed, so a range costs one GET per month spanned.
`03_umami_backfill.ipynb` populated Jan–Aug 2026 from the Umami export.

This section is the read prototype for the eventual owner-gated `GET /stats`.


In [ ]:
from datetime import date, timedelta


def month_keys(site: str, start: date, end: date) -> list[str]:
    """Rollup keys spanning [start, end] — computed, so no listing, no truncation."""
    keys = []
    cursor = start.replace(day=1)
    while cursor <= end:
        keys.append(f"rollup/{site}/{cursor:%Y-%m}.json")
        cursor = (cursor.replace(day=28) + timedelta(days=4)).replace(day=1)
    return keys


def read_range(storage, site: str, start: date, end: date) -> dict[str, dict]:
    """Per-day buckets for [start, end], inclusive, from the monthly rollups."""
    days: dict[str, dict] = {}
    lo, hi = start.isoformat(), end.isoformat()
    for key in month_keys(site, start, end):
        rollup = storage.read(key)
        if not rollup:
            continue
        for day, bucket in rollup.get("days", {}).items():
            if lo <= day <= hi:
                days[day] = bucket
    return dict(sorted(days.items()))


def merge_pages(days: dict[str, dict]) -> dict[str, int]:
    pages: dict[str, int] = {}
    for bucket in days.values():
        for path, count in bucket.get("pages", {}).items():
            pages[path] = pages.get(path, 0) + count
    return dict(sorted(pages.items(), key=lambda item: -item[1]))


### Fixture pass

Same shape as section 1 — validate against hand-computed totals, no credentials.


In [ ]:
local.write(
    "rollup/fixture-site/2026-07.json",
    {
        "site": "fixture-site",
        "month": "2026-07",
        "days": {
            "2026-07-30": {"hits": 3, "pages": {"/": 3}, "source": "umami"},
            "2026-07-31": {"hits": 2, "pages": {"/a/": 2}, "source": "umami"},
        },
    },
)
local.write(
    "rollup/fixture-site/2026-08.json",
    {
        "site": "fixture-site",
        "month": "2026-08",
        "days": {"2026-08-01": {"hits": 4, "pages": {"/": 1, "/a/": 3}, "source": "beacon"}},
    },
)

spanning = read_range(local, "fixture-site", date(2026, 7, 31), date(2026, 8, 1))
assert list(spanning) == ["2026-07-31", "2026-08-01"]  # crosses the month boundary
assert merge_pages(spanning) == {"/a/": 5, "/": 1}
print("range readout OK:", merge_pages(spanning))


### Real readout

`source` is per-day on purpose. Umami filtered bots and sessionised; the beacon
counts every hydration and every client-side navigation, unfiltered. The two
are not the same measurement, so the seam is labelled rather than smoothed over.


In [ ]:
today = datetime.now(timezone.utc).date()
window = read_range(s3, "fretchen.eu", today - timedelta(days=90), today)

by_source: dict[str, int] = {}
for bucket in window.values():
    src = bucket.get("source", "unknown")
    by_source[src] = by_source.get(src, 0) + bucket["hits"]

print(f"{len(window)} days with traffic, {sum(by_source.values())} hits")
print("by source:", by_source)


In [ ]:
# daily sparkline over the window
peak = max((b["hits"] for b in window.values()), default=1)
for day, bucket in window.items():
    bar = "█" * max(1, round(bucket["hits"] * 40 / peak))
    print(f"{day}  {bucket['hits']:>4}  {bar}")


In [ ]:
# top pages across the window
for path, count in list(merge_pages(window).items())[:20]:
    print(f"{count:>6}  {path}")


### Today, from the hourly buckets

The current day is not rolled up yet, so it still comes from `aggregate_day`
above — 24 GETs at most. A `GET /stats` endpoint would do exactly this: rollups
for whole days, hourly for today.


In [ ]:
today_hits, today_pages = aggregate_day(s3, "fretchen.eu", today)
print(f"{today}: {today_hits} hits across {len(today_pages)} pages (live, not yet rolled up)")
